# Colab Setup — Run This First

**Run this notebook ONCE at the start of every Colab session.**

A Colab session is a blank Linux machine. This notebook:
1. Clones your GitHub repo so all your code and data is available
2. Installs the one missing package (`timm`)
3. Mounts Google Drive so trained model weights are saved permanently

> **Why every session?** Colab sessions time out after ~90 min of inactivity. When you reconnect, the machine is wiped clean. Your Drive is persistent though — models saved there survive.

---
## Step 1 — Enable GPU

Before running anything:
1. Click **Runtime** in the top menu
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click Save

This makes training 10-50x faster than CPU.

In [ ]:
# Verify GPU is enabled
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

import os
import subprocess

REPO_URL  = 'https://github.com/jacobfss777/csc3109-g30.git'
REPO_NAME = 'csc3109-g30'
REPO_PATH = f'/content/{REPO_NAME}'

if os.path.exists(REPO_PATH):
    print('Repo already exists. Pulling latest changes...')
    result = subprocess.run(['git', '-C', REPO_PATH, 'pull'], 
                          capture_output=True, text=True)
    if result.returncode != 0:
        print('Error pulling:', result.stderr)
    else:
        print('Pulled successfully.')
else:
    print('Cloning repo (this may take 1-2 minutes)...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_PATH],
                          capture_output=True, text=True)
    if result.returncode != 0:
        print('ERROR cloning repo:')
        print(result.stderr)
        raise RuntimeError(f'Failed to clone {REPO_URL}')
    else:
        print('Clone successful.')

# Verify repo exists
if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(f'Repo not found at {REPO_PATH}')

print(f'\nRepo is at: {REPO_PATH}')
print('Contents:')
for item in sorted(os.listdir(REPO_PATH)):
    print(f'  {item}')

In [ ]:
import os

REPO_URL  = 'https://github.com/jacobfss777/csc3109-g30.git'
REPO_NAME = 'csc3109-g30'
REPO_PATH = f'/content/{REPO_NAME}'

if os.path.exists(REPO_PATH):
    print('Repo already cloned. Pulling latest changes...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_PATH}')

print(f'Repo is at: {REPO_PATH}')
print('Contents:')
for item in sorted(os.listdir(REPO_PATH)):
    print(f'  {item}')

---
## Step 3 — Install Missing Packages

Colab already has PyTorch, torchvision, numpy, matplotlib, sklearn, pandas, PIL.
We only need to install `timm` (used by the Vision Transformer notebook).

In [ ]:
!pip install timm --quiet
print('timm installed.')

---
## Step 4 — Mount Google Drive

Trained model weights (`.pth` files) will be saved to your Google Drive so they persist after the session ends.

A popup will ask you to sign in to your Google account — allow it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder in Drive to store model checkpoints
DRIVE_MODELS = '/content/drive/MyDrive/csc3109_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

# Also create models/ folder inside the repo (used by training scripts)
os.makedirs(f'{REPO_PATH}/models', exist_ok=True)

print(f'Models will be saved to: {REPO_PATH}/models/')
print(f'And backed up to Drive:  {DRIVE_MODELS}/')

---
## Step 5 — Create the Train/Val Split

Run Phase 1's EDA to create `data/train/` and `data/val/` from the raw dataset.

Skip this if the split already exists from a previous session (it's recreated from the repo each time since `data/` is not committed to git).

In [ ]:
import sys
import random
import shutil
from pathlib import Path

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

repo = Path(REPO_PATH)
DATA_ROOT = repo / 'ml training data' / 'set 30'
TRAIN_DIR = repo / 'data' / 'train'
VAL_DIR   = repo / 'data' / 'val'
VAL_PER_CLASS = 100
SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

CLASS_NAMES = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f'Categories: {CLASS_NAMES}')

# Check if split already exists
split_done = all(
    (VAL_DIR / cls).exists() and len(list((VAL_DIR / cls).iterdir())) > 0
    for cls in CLASS_NAMES
)

if split_done:
    print('Split already exists - skipping.')
else:
    print('Creating train/val split...')
    for cls in CLASS_NAMES:
        all_paths = sorted([p for p in (DATA_ROOT / cls).iterdir()
                            if p.suffix.lower() in SUPPORTED_EXTS])
        random.shuffle(all_paths)
        val_paths   = all_paths[:VAL_PER_CLASS]
        train_paths = all_paths[VAL_PER_CLASS:]

        (TRAIN_DIR / cls).mkdir(parents=True, exist_ok=True)
        (VAL_DIR   / cls).mkdir(parents=True, exist_ok=True)

        for p in train_paths:
            shutil.copy2(p, TRAIN_DIR / cls / p.name)
        for p in val_paths:
            shutil.copy2(p, VAL_DIR / cls / p.name)

        print(f'  {cls}: train={len(train_paths)}  val={len(val_paths)}')
    print('Done.')

---
## Setup Complete

Your environment is ready. You can now open and run any training notebook.

**Important:** every training notebook needs to know where the repo is. The path `/content/csc3109-g30` is set automatically when you run the first cell of each notebook (it detects Colab and sets the path for you).

**Workflow for each approach:**
1. Open the notebook (e.g. `03a_custom_cnn.ipynb`)
2. Run all cells
3. When training finishes, run the final backup cell to save the `.pth` to Drive

| Notebook | Estimated GPU time |
|----------|-------------------|
| 03a Custom CNN (30 epochs) | ~8 min |
| 03b ResNet-50 Extraction (20 epochs) | ~5 min |
| 03c ResNet-50 Fine-tune (20 epochs) | ~6 min |
| 03d EfficientNet (30 epochs) | ~8 min |
| 03e ViT (30 epochs) | ~15 min |